In [1]:
import pandas as pd
import re

In [3]:
def clean_player_data(input_csv,output_csv):
    df = pd.read_csv(input_csv)

    # Helper function to safely extract regex groups
    def extract_field(pattern, text):
        match = re.search(pattern, str(text))
        return match.group(1).strip() if match else None

    # Extracting core Stats
    df['Age'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)y\.o\.', x))
    df['DOB'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'\(([A-Z][a-z]{2}\s\d{1,2},\s\d{4})\)', x))
    df['Height_cm'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)cm', x))
    df['Weight_kg'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)kg', x))
    df['Overall_Rating'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)\s*\|\s*Overall rating', x))
    df['Potential_Rating'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)(?:\s*[+-]\d+)?\s*\|\s*Potential', x))
    df['Value_EUR'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(€[\d\.KMB]+)\s*\|\s*Value', x))
    df['Wage_EUR'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(€[\d\.KMB]+)\s*\|\s*Wage', x))
    df['Preferred_Foot'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'Preferred foot\s([A-Za-z]+)', x))

   #ATTRIBUTE EXTRACTION
    #| number Attribute Name |
    attr_pattern = r'\|\s*(\d{1,3})\s+([A-Za-z\s]+?)\s*\|'
    
    all_attributes_list = []
    
    for text in df['Raw_Card_Text']:
        matches = re.findall(attr_pattern, str(text))

        # Building a dictionary for the row, cleaning up spaces in column names
        row_attrs = {attr_name.strip().replace(' ', '_'): int(val) for val, attr_name in matches}
        all_attributes_list.append(row_attrs)
        
    attrs_df = pd.DataFrame(all_attributes_list)     # Converting the list of dictionaries into a new DataFrame

    # Combination and Clean Up
    # Concatenate the dynamic attribute columns to the main DataFrame
    df = pd.concat([df, attrs_df], axis=1)
    
    # Converting numeric columns to actual numeric types for the database
    numeric_cols = ['Age', 'Height_cm', 'Weight_kg', 'Overall_Rating', 'Potential_Rating']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Dropping raw string column
    df_cleaned = df.drop(columns=['Raw_Card_Text'])
    df_cleaned.to_csv(output_csv,index=False)
    display(df_cleaned)
    return df_cleaned



In [4]:

cleaned_df=clean_player_data("data/sofifa/sofifa_raw_data_COMPLETE.csv","data/sofifa/sofifa_cleaned.csv")

,URL,Name,Age,DOB,Height_cm,Weight_kg,Overall_Rating,Potential_Rating,Value_EUR,Wage_EUR,...,Sliding_tackle,GK_Diving,GK_Handling,GK_Kicking,GK_Positioning,GK_Reflexes,Marking,Positioning,Tactical_awareness,Tackling
0,https://sofifa.com/player/272465/caleb-roberts...,Caleb Roberts,19,"Oct 24, 2005",178,74,56,70,€350K,€600,...,42.0,14.0,6.0,13.0,13.0,7.0,NaN,NaN,NaN,NaN
1,https://sofifa.com/player/random?1775801033,Javier Villar del Fraile,22,"Mar 15, 2003",187,69,65,74,€1.6M,€2K,...,67.0,8.0,6.0,8.0,7.0,15.0,NaN,NaN,NaN,NaN
2,https://sofifa.com/player/274482/adrian-ascues...,Adrián Ascues,21,"Nov 15, 2002",181,81,65,73,€1.6M,€12K,...,41.0,8.0,14.0,6.0,11.0,8.0,NaN,NaN,NaN,NaN
3,https://sofifa.com/player/272535/josh-bailey/2...,Josh Bailey,18,"Jun 24, 2005",179,72,52,63,€160K,€500,...,44.0,9.0,11.0,11.0,8.0,10.0,NaN,NaN,NaN,NaN
4,https://sofifa.com/player/271880/nestor-jimene...,Nestor Jiménez,22,"Apr 8, 2003",175,66,61,70,€750K,€4K,...,18.0,10.0,13.0,12.0,6.0,8.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6994,https://sofifa.com/player/272834/joao-pedro-go...,João Pedro Gonçalves Neves,20,"Sep 27, 2004",174,66,87,92,€117.5M,€110K,...,82.0,8.0,7.0,5.0,13.0,13.0,NaN,NaN,NaN,NaN
6995,https://sofifa.com/player/276694/johan-manzamb...,Johan Manzambi,19,"Oct 14, 2005",183,75,75,85,€12M,€18K,...,67.0,8.0,10.0,8.0,13.0,9.0,NaN,NaN,NaN,NaN
6996,https://sofifa.com/player/263765/tom-bischof/2...,Tom Bischof,20,"Jun 28, 2005",180,75,78,86,€30.5M,€41K,...,66.0,9.0,13.0,12.0,13.0,6.0,NaN,NaN,NaN,NaN
6997,https://sofifa.com/player/234826/antonio-marti...,Antonio Martínez López,28,"Jun 30, 1997",187,83,1,76,€7.5M,€31K,...,23.0,11.0,11.0,12.0,7.0,9.0,NaN,NaN,NaN,NaN
